In [1]:
import pyodbc

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=MSI\SQLE22;"
    r"DATABASE=UFC;"
    r"Trusted_Connection=yes;"
)

print("CONNECTED")

CONNECTED


In [2]:
import pandas as pd

query = 'select * from fights'

data = pd.read_sql(query,conn)

C:\Users\mplan\AppData\Local\Temp\ipykernel_14492\271505560.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query,conn)


In [3]:
data.head(1)

,fight_id,fight_url,fighter_1,fighter_2,winner,weight_class,method,round_num,fight_time,event_name,...,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub,fighter1_id,fighter2_id
0,1,http://ufcstats.com/fight-details/4acab67848e7...,Scott Morris,Sean Daugherty,Scott Morris,Open Weight,SUB Guillotine Choke,1,0:20,UFC 2: No Way Out,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,10.0,1.0


In [4]:
data.isnull().sum()

fight_id             0
fight_url            0
fighter_1            0
fighter_2            0
winner               0
weight_class         0
method               0
round_num            0
fight_time           0
event_name           0
event_id             0
fighter1_kd         21
fighter2_kd         21
fighter1_strikes    21
fighter2_strikes    21
fighter1_td         21
fighter2_td         21
fighter1_sub        21
fighter2_sub        21
fighter1_id          5
fighter2_id         13
dtype: int64

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8701 entries, 0 to 8700
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   fight_id          8701 non-null   int64  
 1   fight_url         8701 non-null   str    
 2   fighter_1         8701 non-null   str    
 3   fighter_2         8701 non-null   str    
 4   winner            8701 non-null   str    
 5   weight_class      8701 non-null   str    
 6   method            8701 non-null   str    
 7   round_num         8701 non-null   int64  
 8   fight_time        8701 non-null   str    
 9   event_name        8701 non-null   str    
 10  event_id          8701 non-null   int64  
 11  fighter1_kd       8680 non-null   float64
 12  fighter2_kd       8680 non-null   float64
 13  fighter1_strikes  8680 non-null   float64
 14  fighter2_strikes  8680 non-null   float64
 15  fighter1_td       8680 non-null   float64
 16  fighter2_td       8680 non-null   float64
 17  fighte

In [6]:
# οι στήλες που πρέπει να πειράξω είναι οι εξής : winner   weight_class method    fight_time  
# οι στήλες που πρέπει να ρίξω είναι οι εξής :  fighter1 , fighter2  , fight_url , event_name

In [12]:
# ξεκινάω δίνοντας id στον winner
data.head(1)

,fight_id,fight_url,fighter_1,fighter_2,winner,weight_class,method,round_num,fight_time,event_name,...,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub,fighter1_id,fighter2_id,winner_id
0,1,http://ufcstats.com/fight-details/4acab67848e7...,Scott Morris,Sean Daugherty,Scott Morris,Open Weight,SUB Guillotine Choke,1,0:20,UFC 2: No Way Out,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,10.0,1.0,10.0


In [9]:
import numpy as np 
data['winner_id'] = np.where(
    data['fighter_1'] == data['winner'],
    data['fighter1_id'],
    data['fighter2_id']
)

In [13]:
# τώρα θα κάνω mapping τα weight_class 

weight_class_mapping = {}

for i, wc in enumerate(data['weight_class'].unique()):
    weight_class_mapping[wc] = i


data['weight_class_encoded'] = data['weight_class'].map(weight_class_mapping)

In [15]:
# τώρα το ίδιο σε μέθοδο νίκης 

In [25]:
# θα δώσω και μία βασική 

data['method_type'] = np.select(
    [
        data['method'].str.contains('SUB', case=False, na=False),
        data['method'].str.contains('KO/TKO', case=False, na=False),
        data['method'].str.contains('DEC', case=False, na=False)

    ],
    [
        'SUB',
        'KO/TKO',
        'DEC'
    ],
    default='OTHER'
)

In [22]:
# πολύ καλά πάμε τώρα να κάνουμε μαπινγκ και τα δυο και αργότερα βλέπουμε τι κρατάω ή οχι 

In [27]:
# τώρα θα κάνω mapping τα weight_class 

method_detailed = {}

for i, method in enumerate(data['method'].unique()):
    method_detailed[method] = i


data['method_detailed_encoded'] = data['method'].map(method_detailed)




method_basic = {}

for i, method_b in enumerate(data['method_type'].unique()):
    method_basic[method_b] = i


data['method_basic_encoded'] = data['method_type'].map(method_basic)


In [29]:
data.tail(2)

,fight_id,fight_url,fighter_1,fighter_2,winner,weight_class,method,round_num,fight_time,event_name,...,fighter2_td,fighter1_sub,fighter2_sub,fighter1_id,fighter2_id,winner_id,weight_class_encoded,method_type,method_detailed_encoded,method_basic_encoded
8699,8700,http://ufcstats.com/fight-details/fc1266e2892e...,Dooho Choi,Daniel Santos,Dooho Choi,Featherweight,KO/TKO Punch,2,4:29,UFC Fight Night: Allen vs. Costa,...,0.0,0.0,0.0,1979.0,2183.0,1979.0,8,KO/TKO,26,1
8700,8701,http://ufcstats.com/fight-details/e4aa60812489...,Arnold Allen,Melquizael Costa,Arnold Allen,Featherweight,U-DEC,5,5:00,UFC Fight Night: Allen vs. Costa,...,0.0,0.0,0.0,2322.0,2393.0,2322.0,8,DEC,25,3


In [30]:
# έχουμε ένα τελευταιο που πρέπει να γίνει το fight time και θα το μετατρέψω όλο σε second

In [45]:
x_list = []
y_list = []

for t in data['fight_time']:
    x, y = t.split(':')
    x_list.append(x)
    y_list.append(y)

final_time = [(int(x_list[i])*60)+int(y_list[i]) for i in range(len(x_list))]
data['total_seconds'] = final_time

In [46]:
# ok πάμε να ρίξουμε τα αχρείαστα columns και να κάνουμε τα nulls

In [47]:
data.head(1)

,fight_id,fight_url,fighter_1,fighter_2,winner,weight_class,method,round_num,fight_time,event_name,...,fighter1_sub,fighter2_sub,fighter1_id,fighter2_id,winner_id,weight_class_encoded,method_type,method_detailed_encoded,method_basic_encoded,total_seconds
0,1,http://ufcstats.com/fight-details/4acab67848e7...,Scott Morris,Sean Daugherty,Scott Morris,Open Weight,SUB Guillotine Choke,1,0:20,UFC 2: No Way Out,...,1.0,0.0,10.0,1.0,10.0,0,SUB,0,0,20


In [48]:
data = data.drop(columns=['fight_url','fighter_1','fighter_2','winner','weight_class','method','event_name','fight_time','method_type'])

In [49]:
data.head(1)

,fight_id,round_num,event_id,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub,fighter1_id,fighter2_id,winner_id,weight_class_encoded,method_detailed_encoded,method_basic_encoded,total_seconds
0,1,1,1,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,10.0,1.0,10.0,0,0,0,20


In [50]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8701 entries, 0 to 8700
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   fight_id                 8701 non-null   int64  
 1   round_num                8701 non-null   int64  
 2   event_id                 8701 non-null   int64  
 3   fighter1_kd              8680 non-null   float64
 4   fighter2_kd              8680 non-null   float64
 5   fighter1_strikes         8680 non-null   float64
 6   fighter2_strikes         8680 non-null   float64
 7   fighter1_td              8680 non-null   float64
 8   fighter2_td              8680 non-null   float64
 9   fighter1_sub             8680 non-null   float64
 10  fighter2_sub             8680 non-null   float64
 11  fighter1_id              8696 non-null   float64
 12  fighter2_id              8688 non-null   float64
 13  winner_id                8696 non-null   float64
 14  weight_class_encoded     8701 non-n

In [51]:
#πάμε να τα κάνουμε int σε πρώτη φάση 

cols = [
    'fighter1_kd','fighter2_kd',
    'fighter1_strikes','fighter2_strikes',
    'fighter1_td','fighter2_td',
    'fighter1_sub','fighter2_sub',
    'fighter1_id','fighter2_id','winner_id'
]

data[cols] = data[cols].astype('Int64')

In [53]:
data.isnull().sum()

fight_id                    0
round_num                   0
event_id                    0
fighter1_kd                21
fighter2_kd                21
fighter1_strikes           21
fighter2_strikes           21
fighter1_td                21
fighter2_td                21
fighter1_sub               21
fighter2_sub               21
fighter1_id                 5
fighter2_id                13
winner_id                   5
weight_class_encoded        0
method_detailed_encoded     0
method_basic_encoded        0
total_seconds               0
dtype: int64

In [55]:
#έχουμε nulls θα τα ρίξουμε γιατί είναι πολύ μικρό ποσοστό των δεδομένων μας 
data.shape

(8701, 18)

In [56]:
data = data.dropna()

In [57]:
data.isnull().sum().sum()

np.int64(0)

In [58]:
data.shape

(8680, 18)

In [61]:
data.head(10)

,fight_id,round_num,event_id,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub,fighter1_id,fighter2_id,winner_id,weight_class_encoded,method_detailed_encoded,method_basic_encoded,total_seconds
0,1,1,1,0,0,1,0,1,0,1,0,10,1,10,0,0,0,20
1,2,1,1,0,0,1,1,0,0,1,0,30,2,30,0,0,0,58
2,3,1,1,0,0,11,4,1,0,0,0,13,3,13,0,1,1,733
3,4,1,1,0,0,2,0,1,0,3,0,5,4,5,0,2,0,292
4,5,1,1,0,0,8,2,0,1,0,1,12,6,12,0,3,1,170
5,6,1,1,0,0,4,1,1,0,1,0,38,7,38,0,4,0,591
6,7,1,1,0,0,3,0,0,1,5,0,138,8,138,0,5,0,401
7,8,1,1,0,0,2,3,1,0,2,0,274,9,274,0,6,0,308
8,9,1,1,0,0,13,0,0,0,0,0,30,10,30,0,7,1,30
9,10,1,1,1,0,13,4,0,0,1,0,13,11,13,0,8,0,187


In [59]:
# αποθηκέυω τα mappings κάνω export τα δεδομένα και είμαι οκ απο εδώ

import json

with open(r"C:\Users\mplan\Desktop\ufc\MAPPINGS\weight_class_mapping.json", "w") as f:
    json.dump(weight_class_mapping, f, indent=4)

with open(r"C:\Users\mplan\Desktop\ufc\MAPPINGS\method_detailed.json", "w") as f:
    json.dump(method_detailed, f, indent=4)

with open(r"C:\Users\mplan\Desktop\ufc\MAPPINGS\method_basic.json", "w") as f:
    json.dump(method_basic, f, indent=4)

In [62]:
data.to_csv(r'C:\Users\mplan\Desktop\ufc\TRAINING_DATA\fights_training.csv',index=False)